# 🍃 Coffee Leaf Multi-Task CNN (Mobile/AR-Ready)
## Skripsi: Implementasi Model Multi-Task CNN Lightweight untuk Diagnosis Penyakit Daun Kopi Berbasis Mobile

Notebook ini melatih model deep learning **Multi-Task CNN (3 Output)**:
1. **Task 1: Diagnosis Penyakit** (5 Kelas: Healthy, Miner, Leaf rust, Phoma, Cercospora)
2. **Task 2: Tingkat Keparahan** (Regresi Kontinu 0.0 - 1.0)
3. **Task 3: Lokalisasi Bounding Box** (Regresi `[cx, cy, w, h]` untuk Overlay AR di Unity)

### Kandidat Backbone & Skema Eksperimen (Matriks 2 x 2):
- **Backbone:** MobileNetV2 vs. EfficientNet-B0
- **Skema:** Multi-Task (1 shared backbone + 3 head) vs. Single-Task (baseline terpisah per task)
- **Export Target:** ONNX untuk Unity Sentis lengkap dengan `model_meta.json`

> ⚠️ **PENTING:** Aktifkan GPU T4 sebelum mulai:
> Menu Colab: `Runtime` -> `Change runtime type` -> Pilih **T4 GPU** -> `Save`.

## 1. Hubungkan ke Google Drive & Ekstrak Dataset

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Ekstrak file zip penelitian (sesuaikan path file zip Anda di Google Drive)
!unzip -q "/content/drive/MyDrive/Skripsi_Kopi/penelitian_kopi.zip" -d /content/

# Periksa ketersediaan GPU
!nvidia-smi

## 2. Masuk ke Folder Project & Verifikasi Lingkungan

In [ ]:
import os
import torch

%cd /content/coffee-leaf-multitask-cnn/classification
!pwd

print('PyTorch Version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name :', torch.cuda.get_device_name(0))
else:
    print('PERINGATAN: GPU belum aktif! Silakan ubah runtime type ke T4 GPU.')

## 3. Siapkan Dataset Gabungan (JMUBEN + JMUBEN V2 + BRACOL)
Jalankan cell ini jika file `dataset_combined.csv` atau `dataset_combined_balanced.csv` belum dibuat di Colab.

In [ ]:
from prepare_combined_dataset import prepare_combined_dataset

# Buat dataset balanced (1000 gambar/kelas) agar training cepat & seimbang
prepare_combined_dataset(
    workspace_root="/content",
    output_csv="dataset/dataset_combined_balanced.csv",
    sample_per_class=1000
)

## 4. Eksperimen 1: Multi-Task CNN dengan MobileNetV2
Melatih model 3-head (Penyakit + Keparahan + Bbox) dengan backbone **MobileNetV2**.

In [ ]:
!python main.py --train \
  --backbone mobilenet \
  --mode multi \
  --dataset combined \
  --csv_file dataset/dataset_combined_balanced.csv \
  --epochs 20 \
  --batch_size 32 \
  --lr 0.01 \
  --optimizer sgd \
  --lambda1 1.0 \
  --lambda2 1.0 \
  --lambda3 1.0 \
  --experiment_name exp_multitask_mobilenet

## 5. Eksperimen 2: Multi-Task CNN dengan EfficientNet-B0
Melatih model 3-head dengan backbone **EfficientNet-B0** untuk perbandingan performa & ukuran.

In [ ]:
!python main.py --train \
  --backbone efficientnet \
  --mode multi \
  --dataset combined \
  --csv_file dataset/dataset_combined_balanced.csv \
  --epochs 20 \
  --batch_size 32 \
  --lr 0.01 \
  --optimizer sgd \
  --lambda1 1.0 \
  --lambda2 1.0 \
  --lambda3 1.0 \
  --experiment_name exp_multitask_efficientnet

## 6. Eksperimen 3 (Opsional): Baseline Single-Task (Untuk Bab 4 Skripsi)
Melatih model single-task untuk membuktikan keunggulan multi-task vs. single-task.

In [ ]:
# Single-Task: Diagnosis Penyakit saja (MobileNetV2)
!python main.py --train \
  --backbone mobilenet \
  --mode single \
  --task disease \
  --dataset combined \
  --csv_file dataset/dataset_combined_balanced.csv \
  --epochs 20 \
  --batch_size 32 \
  --experiment_name exp_singletask_disease_mobilenet

## 7. Export Model ke Format ONNX untuk Unity Sentis & Verifikasi Output
Cell ini mengonversi model Multi-Task ke ONNX sesuai Kontrak Output Section 3.6:
- Output 1: `disease_probs` [1, 5] (sudah Softmax)
- Output 2: `severity` [1, 1] (0.0 - 1.0)
- Output 3: `bbox` [1, 4] (`[cx, cy, w, h]`)
- Menghasilkan file metadata: `model_unity_meta.json`

In [ ]:
# Export MobileNetV2 Multi-Task
!python export_onnx.py \
  --weights results/exp_multitask_mobilenet/best_weights.pth \
  --output results/model_unity_mobilenet.onnx \
  --backbone mobilenet \
  --opset 12

# Export EfficientNet-B0 Multi-Task
!python export_onnx.py \
  --weights results/exp_multitask_efficientnet/best_weights.pth \
  --output results/model_unity_efficientnet.onnx \
  --backbone efficientnet \
  --opset 12

# Verifikasi numerik kesesuaian PyTorch vs ONNX Runtime
!python verify_export.py \
  --onnx results/model_unity_mobilenet.onnx \
  --weights results/exp_multitask_mobilenet/best_weights.pth \
  --backbone mobilenet

## 8. Download Model ONNX & File Metadata untuk Unity AR

In [ ]:
from google.colab import files

# Download model ONNX MobileNetV2 dan file metadata untuk Unity
files.download('results/model_unity_mobilenet.onnx')
files.download('results/model_unity_mobilenet_meta.json')

# Download log training untuk grafik skripsi
files.download('results/exp_multitask_mobilenet/train_logs.json')